# Multi-Seed Safari+WavLM Ensemble (Stepwise, Challenge Split)

This notebook runs a harder evaluation protocol with unseen English fake generators in validation/test.
It uses `RawCh_*` CSVs and a shared embedding root `WavLM_embeddings_unified`.

Suggested order:
1. Run setup/config cells.
2. Set `CURRENT_SEED`.
3. Run train + predict cells for that seed.
4. Repeat for remaining seeds.
5. Run the aggregation cell.


In [7]:
#1
from pathlib import Path
from argparse import Namespace
import csv
import json
import sys
from typing import Dict, List, Tuple

import torch

ROOT = Path.cwd().resolve()
if not (ROOT / 'safari_wavlm_ensemble_train.py').exists():
    if (ROOT.parent / 'safari_wavlm_ensemble_train.py').exists():
        ROOT = ROOT.parent
    else:
        # fallback search upward
        for parent in ROOT.parents:
            if (parent / 'safari_wavlm_ensemble_train.py').exists():
                ROOT = parent
                break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from safari_wavlm_ensemble_train import train as train_ensemble
from predict_safari_wavlm_ensemble import (
    device_from_arg,
    load_ckpt,
    build_model,
    run_batch,
)

ARTIFACT_ROOT = ROOT / 'artifacts' / 'multiseed_stepwise_challenge'
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('ARTIFACT_ROOT:', ARTIFACT_ROOT)
print('python:', sys.executable)



ROOT: C:\Users\sovan\Documents\VS Code\SDP
ARTIFACT_ROOT: C:\Users\sovan\Documents\VS Code\SDP\artifacts\multiseed_stepwise_challenge
python: c:\Users\sovan\Documents\VS Code\SDP\.venv\Scripts\python.exe


In [8]:
# Global config (strict split + unified WavLM embeddings)
SEEDS = [17, 42, 77, 123, 202]

TRAIN_CSV = ROOT / 'RawCh_train' / 'RawCh_train' / 'balanced_index.csv'
VAL_CSV = ROOT / 'RawCh_val' / 'RawCh_val' / 'balanced_index.csv'
TEST_CSV = ROOT / 'RawCh_test' / 'RawCh_test' / 'balanced_index.csv'


def has_npy_files(path: Path) -> bool:
    if not path.exists():
        return False
    return any(path.rglob('*.npy'))

WAVLM_UNIFIED_ROOT = ROOT / 'WavLM_embeddings_unified'

# Strict split uses a shared embedding root, so each split can resolve its feature_path.
TRAIN_BASE_ROOT = WAVLM_UNIFIED_ROOT
VAL_BASE_ROOT = WAVLM_UNIFIED_ROOT
TEST_BASE_ROOT = WAVLM_UNIFIED_ROOT
TRAIN_WAVLM_ROOT = WAVLM_UNIFIED_ROOT
VAL_WAVLM_ROOT = WAVLM_UNIFIED_ROOT
TEST_WAVLM_ROOT = WAVLM_UNIFIED_ROOT

missing = []
for name, pth in [
    ('TRAIN_CSV', TRAIN_CSV),
    ('VAL_CSV', VAL_CSV),
    ('TEST_CSV', TEST_CSV),
    ('WAVLM_UNIFIED_ROOT', WAVLM_UNIFIED_ROOT),
]:
    if not pth.exists():
        missing.append(name)

if WAVLM_UNIFIED_ROOT.exists() and not has_npy_files(WAVLM_UNIFIED_ROOT):
    missing.append('WAVLM_UNIFIED_ROOT (no .npy found)')

if missing:
    raise FileNotFoundError('Fix these paths before training: ' + ', '.join(sorted(set(missing))))

TRAIN_BASE_ARGS = {
    'project_root': str(ROOT),
    'train_split': 'RawCh_train',
    'val_split': 'RawCh_val',
    'test_split': 'RawCh_test',
    'train_csv': str(TRAIN_CSV),
    'val_csv': str(VAL_CSV),
    'test_csv': str(TEST_CSV),
    'train_feature_dir': str(TRAIN_BASE_ROOT),
    'val_feature_dir': str(VAL_BASE_ROOT),
    'test_feature_dir': str(TEST_BASE_ROOT),
    'wavlm_train_feature_dir': str(TRAIN_WAVLM_ROOT),
    'wavlm_val_feature_dir': str(VAL_WAVLM_ROOT),
    'wavlm_test_feature_dir': str(TEST_WAVLM_ROOT),
    'out_dir': '',  # filled per seed
    'epochs': 60,
    'batch_size': 512,
    'num_workers': 4,
    'hidden_dim': 384,
    'language_emb_dim': 32,
    'lr': 3e-4,
    'max_lr': 2e-3,
    'weight_decay': 1e-4,
    'patience': 10,
    'view_noise_std': 0.01,
    'view_drop_prob': 0.03,
    'stats_samples': 50000,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'amp': True,
    'seed': 42,
}

print('Seeds:', SEEDS)
print('Train CSV:', TRAIN_CSV)
print('Val CSV:', VAL_CSV)
print('Test CSV:', TEST_CSV)
print('Train epochs:', TRAIN_BASE_ARGS['epochs'])
print('Early stopping patience:', TRAIN_BASE_ARGS['patience'])
print('Train device:', TRAIN_BASE_ARGS['device'])
print('Unified embedding root:', TRAIN_BASE_ARGS['train_feature_dir'])


Seeds: [17, 42, 77, 123, 202]
Train CSV: C:\Users\sovan\Documents\VS Code\SDP\RawCh_train\RawCh_train\balanced_index.csv
Val CSV: C:\Users\sovan\Documents\VS Code\SDP\RawCh_val\RawCh_val\balanced_index.csv
Test CSV: C:\Users\sovan\Documents\VS Code\SDP\RawCh_test\RawCh_test\balanced_index.csv
Train epochs: 60
Early stopping patience: 10
Train device: cpu
Unified embedding root: C:\Users\sovan\Documents\VS Code\SDP\WavLM_embeddings_unified


In [9]:
#3
def parse_label(v: str) -> int:
    return 1 if v.strip().lower() == 'fake' else 0


def auc_score(labels: List[int], scores: List[float]) -> float:
    n = len(labels)
    order = sorted(range(n), key=lambda i: scores[i])
    ranks = [0.0] * n
    i = 0
    rank = 1
    while i < n:
        j = i
        while j + 1 < n and scores[order[j + 1]] == scores[order[i]]:
            j += 1
        avg_rank = (rank + (rank + (j - i))) / 2.0
        for k in range(i, j + 1):
            ranks[order[k]] = avg_rank
        rank += j - i + 1
        i = j + 1

    pos = [idx for idx, y in enumerate(labels) if y == 1]
    neg = [idx for idx, y in enumerate(labels) if y == 0]
    n_pos = len(pos)
    n_neg = len(neg)
    if n_pos == 0 or n_neg == 0:
        return float('nan')

    sum_pos_ranks = sum(ranks[idx] for idx in pos)
    return (sum_pos_ranks - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)


def f1_score(labels: List[int], preds: List[int]) -> float:
    tp = sum(1 for y, p in zip(labels, preds) if y == 1 and p == 1)
    fp = sum(1 for y, p in zip(labels, preds) if y == 0 and p == 1)
    fn = sum(1 for y, p in zip(labels, preds) if y == 1 and p == 0)
    if tp == 0:
        return 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    return 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0


def accuracy(labels: List[int], preds: List[int]) -> float:
    return sum(1 for y, p in zip(labels, preds) if y == p) / max(1, len(labels))


def read_prediction_csv(path: Path) -> Dict[str, Tuple[int, float]]:
    out: Dict[str, Tuple[int, float]] = {}
    with path.open('r', encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            feature_path = row['feature_path'].strip()
            label = parse_label(row['label'])
            prob = float(row['fake_probability'])
            out[feature_path] = (label, prob)
    return out


def aggregate_mean_prob(pred_csv_paths: List[Path]):
    tables = [read_prediction_csv(p) for p in pred_csv_paths]
    shared_keys = sorted(set.intersection(*(set(t.keys()) for t in tables)))

    labels: List[int] = []
    probs: List[float] = []
    for k in shared_keys:
        y = tables[0][k][0]
        p = sum(t[k][1] for t in tables) / len(tables)
        labels.append(y)
        probs.append(p)
    return shared_keys, labels, probs


def best_threshold(labels: List[int], probs: List[float]) -> Tuple[float, float]:
    best_t = 0.5
    best_f1 = -1.0
    for i in range(181):
        t = 0.05 + i * 0.005
        preds = [1 if p >= t else 0 for p in probs]
        f1 = f1_score(labels, preds)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    return best_t, best_f1


In [10]:
#4
PREDICT_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PRED_DEVICE_OBJ = device_from_arg(PREDICT_DEVICE)
print('Prediction device:', PRED_DEVICE_OBJ)


def seed_dir(seed: int) -> Path:
    d = ARTIFACT_ROOT / f'seed_{seed}'
    d.mkdir(parents=True, exist_ok=True)
    return d


def train_one_seed(seed: int) -> Path:
    out_dir = seed_dir(seed)
    cfg = dict(TRAIN_BASE_ARGS)
    cfg['seed'] = seed
    cfg['out_dir'] = str(out_dir)
    args = Namespace(**cfg)

    print(f'\n=== Training seed {seed} ===')
    train_ensemble(args)
    print(f'Finished seed {seed}:', out_dir)
    return out_dir


def predict_seed_split(seed: int, split: str) -> Path:
    d = seed_dir(seed)
    model_path = d / 'best_safari_wavlm_ensemble.pt'
    if not model_path.exists():
        raise FileNotFoundError(f'Model not found for seed {seed}: {model_path}')

    ckpt = load_ckpt(model_path, PRED_DEVICE_OBJ)
    model = build_model(ckpt, PRED_DEVICE_OBJ)

    if split == 'val':
        input_csv = VAL_CSV
        base_root = VAL_BASE_ROOT
        wavlm_root = VAL_WAVLM_ROOT
        output_csv = d / 'val_predictions.csv'
    elif split == 'test':
        input_csv = TEST_CSV
        base_root = TEST_BASE_ROOT
        wavlm_root = TEST_WAVLM_ROOT
        output_csv = d / 'test_predictions.csv'
    else:
        raise ValueError("split must be 'val' or 'test'")

    run_batch(
        model=model,
        ckpt=ckpt,
        input_csv=input_csv,
        base_feature_root=base_root,
        wavlm_feature_root=wavlm_root,
        output_csv=output_csv,
        device=PRED_DEVICE_OBJ,
    )
    print(f'Finished {split} prediction for seed {seed}:', output_csv)
    return output_csv


Prediction device: cpu


In [11]:
#5 # Choose one seed to run step-by-step.
CURRENT_SEED = SEEDS[0]
CURRENT_SEED


17

In [12]:
#6 # 1) Train this seed
current_dir = train_one_seed(CURRENT_SEED)
current_dir



=== Training seed 17 ===


Dataset setup:
  Train CSV: C:\Users\sovan\Documents\VS Code\SDP\RawCh_train\RawCh_train\balanced_index.csv
  Val CSV:   C:\Users\sovan\Documents\VS Code\SDP\RawCh_val\RawCh_val\balanced_index.csv
  Test CSV:  C:\Users\sovan\Documents\VS Code\SDP\RawCh_test\RawCh_test\balanced_index.csv
  Base dim: 768 | WavLM dim: 768
  Device: cpu | AMP=off
Loaded samples:
  Train=3789 (base-missing=0, wavlm-fallback=0, unk-lang=0)
  Val=1605 (base-missing=0, wavlm-fallback=0, unk-lang=0)
  Test=2815 (base-missing=0, wavlm-fallback=0, unk-lang=0)

Epoch 1/60


  train_loss=0.42916 val_loss=0.72153 val_auc=0.68988 val_f1=0.71125 threshold=0.100 lr=0.0004187
  Saved best model: C:\Users\sovan\Documents\VS Code\SDP\artifacts\multiseed_stepwise_challenge\seed_17\best_safari_wavlm_ensemble.pt

Epoch 2/60


  train_loss=0.06650 val_loss=1.87402 val_auc=0.71656 val_f1=0.50616 threshold=0.050 lr=0.0007415
  Saved best model: C:\Users\sovan\Documents\VS Code\SDP\artifacts\multiseed_stepwise_challenge\seed_17\best_safari_wavlm_ensemble.pt

Epoch 3/60


  train_loss=0.07166 val_loss=1.04477 val_auc=0.90123 val_f1=0.76589 threshold=0.050 lr=0.0011784
  Saved best model: C:\Users\sovan\Documents\VS Code\SDP\artifacts\multiseed_stepwise_challenge\seed_17\best_safari_wavlm_ensemble.pt

Epoch 4/60


...

  train_loss=0.00024 val_loss=0.47712 val_auc=0.96365 val_f1=0.89899 threshold=0.085 lr=0.0016814
  No improvement (8/10)

Epoch 21/60


  train_loss=0.00168 val_loss=0.50377 val_auc=0.97130 val_f1=0.90280 threshold=0.050 lr=0.0016377
  No improvement (9/10)

Epoch 22/60


  train_loss=0.00472 val_loss=0.44566 val_auc=0.97016 val_f1=0.91040 threshold=0.145 lr=0.0015919
  No improvement (10/10)
Early stopping.



Final metrics:
  Best epoch: 12
  Val AUC/F1: 0.97843 / 0.90528
  Test AUC/F1:0.95175 / 0.84514
  Model: C:\Users\sovan\Documents\VS Code\SDP\artifacts\multiseed_stepwise_challenge\seed_17\best_safari_wavlm_ensemble.pt
  Metrics: C:\Users\sovan\Documents\VS Code\SDP\artifacts\multiseed_stepwise_challenge\seed_17\metrics_safari_wavlm_ensemble.json
Finished seed 17: C:\Users\sovan\Documents\VS Code\SDP\artifacts\multiseed_stepwise_challenge\seed_17


WindowsPath('C:/Users/sovan/Documents/VS Code/SDP/artifacts/multiseed_stepwise_challenge/seed_17')

In [13]:
#7 # 2) Predict validation split for this seed
val_pred_path = predict_seed_split(CURRENT_SEED, 'val')
val_pred_path


Wrote predictions: C:\Users\sovan\Documents\VS Code\SDP\artifacts\multiseed_stepwise_challenge\seed_17\val_predictions.csv
Rows: 1605 | Threshold: 0.050
Finished val prediction for seed 17: C:\Users\sovan\Documents\VS Code\SDP\artifacts\multiseed_stepwise_challenge\seed_17\val_predictions.csv


WindowsPath('C:/Users/sovan/Documents/VS Code/SDP/artifacts/multiseed_stepwise_challenge/seed_17/val_predictions.csv')

In [14]:
#8 # 3) Predict test split for this seed
test_pred_path = predict_seed_split(CURRENT_SEED, 'test')
test_pred_path


Wrote predictions: C:\Users\sovan\Documents\VS Code\SDP\artifacts\multiseed_stepwise_challenge\seed_17\test_predictions.csv
Rows: 2815 | Threshold: 0.050
Finished test prediction for seed 17: C:\Users\sovan\Documents\VS Code\SDP\artifacts\multiseed_stepwise_challenge\seed_17\test_predictions.csv


WindowsPath('C:/Users/sovan/Documents/VS Code/SDP/artifacts/multiseed_stepwise_challenge/seed_17/test_predictions.csv')

In [15]:
#9 # Optional convenience cell: run all seeds in one go.
# You can keep this False and run seed-by-seed using the cells above.
RUN_ALL_SEEDS = False

if RUN_ALL_SEEDS:
    for seed in SEEDS:
        train_one_seed(seed)
        predict_seed_split(seed, 'val')
        predict_seed_split(seed, 'test')


In [16]:
#10 # Show which seeds have complete predictions
completed = []
for seed in SEEDS:
    d = seed_dir(seed)
    if (d / 'val_predictions.csv').exists() and (d / 'test_predictions.csv').exists():
        completed.append(seed)

print('Completed seeds:', completed)
print('Pending seeds:', [s for s in SEEDS if s not in completed])


Completed seeds: [17]
Pending seeds: [42, 77, 123, 202]


In [17]:
#11 # Aggregate completed seeds into one ensemble result
completed_seed_dirs = [
    seed_dir(s)
    for s in SEEDS
    if (seed_dir(s) / 'val_predictions.csv').exists() and (seed_dir(s) / 'test_predictions.csv').exists()
]

if len(completed_seed_dirs) < 2:
    CAN_AGGREGATE = False
    print('Need at least 2 completed seeds to compute ensemble metrics.')
    print('Completed so far:', [d.name for d in completed_seed_dirs])
    print('Run more seeds, then rerun this cell.')
else:
    CAN_AGGREGATE = True
    val_pred_paths = [d / 'val_predictions.csv' for d in completed_seed_dirs]
    test_pred_paths = [d / 'test_predictions.csv' for d in completed_seed_dirs]

    val_keys, val_labels, val_probs = aggregate_mean_prob(val_pred_paths)
    test_keys, test_labels, test_probs = aggregate_mean_prob(test_pred_paths)

    threshold, val_f1 = best_threshold(val_labels, val_probs)
    val_preds = [1 if p >= threshold else 0 for p in val_probs]
    test_preds = [1 if p >= threshold else 0 for p in test_probs]

    val_metrics = {
        'roc_auc': auc_score(val_labels, val_probs),
        'f1': val_f1,
        'accuracy': accuracy(val_labels, val_preds),
        'threshold': threshold,
    }
    test_metrics = {
        'roc_auc': auc_score(test_labels, test_probs),
        'f1': f1_score(test_labels, test_preds),
        'accuracy': accuracy(test_labels, test_preds),
        'threshold': threshold,
    }

    print('Val metrics:', val_metrics)
    print('Test metrics:', test_metrics)



Need at least 2 completed seeds to compute ensemble metrics.
Completed so far: ['seed_17']
Run more seeds, then rerun this cell.


In [18]:
#12 # Save ensemble predictions + summary
if not globals().get('CAN_AGGREGATE', False):
    print('Skipping save: ensemble metrics are not ready yet. Finish at least 2 seeds and rerun cell 11.')
else:
    ensemble_dir = ARTIFACT_ROOT / 'ensemble_predictions'
    ensemble_dir.mkdir(parents=True, exist_ok=True)

    val_out = ensemble_dir / 'val_ensemble_predictions.csv'
    test_out = ensemble_dir / 'test_ensemble_predictions.csv'
    summary_out = ARTIFACT_ROOT / 'multiseed_ensemble_summary.json'

    with val_out.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['feature_path', 'label', 'fake_probability', 'predicted_label'])
        writer.writeheader()
        for k, y, p in zip(val_keys, val_labels, val_probs):
            writer.writerow({
                'feature_path': k,
                'label': 'fake' if y == 1 else 'real',
                'fake_probability': f'{p:.6f}',
                'predicted_label': 'fake' if p >= threshold else 'real',
            })

    with test_out.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['feature_path', 'label', 'fake_probability', 'predicted_label'])
        writer.writeheader()
        for k, y, p in zip(test_keys, test_labels, test_probs):
            writer.writerow({
                'feature_path': k,
                'label': 'fake' if y == 1 else 'real',
                'fake_probability': f'{p:.6f}',
                'predicted_label': 'fake' if p >= threshold else 'real',
            })

    summary = {
        'seeds_requested': SEEDS,
        'completed_seed_dirs': [str(d) for d in completed_seed_dirs],
        'val_metrics': val_metrics,
        'test_metrics': test_metrics,
        'val_predictions': str(val_out),
        'test_predictions': str(test_out),
    }
    summary_out.write_text(json.dumps(summary, indent=2), encoding='utf-8')

    print('Saved:', val_out)
    print('Saved:', test_out)
    print('Saved:', summary_out)



Skipping save: ensemble metrics are not ready yet. Finish at least 2 seeds and rerun cell 11.
